# Step 4: XGBoost Dual-Objective Matcher
Trains two XGBoost models:
- **Regressor**: predicts detour distance (km)
- **Classifier**: predicts acceptance probability

Includes grid search hyperparameter tuning, 5-fold cross-validation, and saves feature importance + ROC curve plots.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Enable inline plots for notebook display
%matplotlib inline

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import pandas as pd
import numpy as np

from src.models.xgb_matching import MultiObjectiveMatcher
print("Imported MultiObjectiveMatcher from src/models/xgb_matching.py")

Imported MultiObjectiveMatcher from src/models/xgb_matching.py


In [3]:
# Run the XGBoost dual-objective matcher
print("Loading simulated candidate pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
matcher = MultiObjectiveMatcher()
X_test, yr_test, yc_test, pred_detour, pred_match = matcher.train(df)

Loading simulated candidate pairs...
Training on 38102 samples with 9 features.
Driver-level split: 30425 train pairs, 7677 test pairs.
No driver_id overlap between train and test: True

Running grid search for classifier hyperparameters...


Best classifier params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
Class balance — pos: 13273, neg: 17152, scale_pos_weight: 1.292

Running 5-fold cross-validation...


Classifier CV Accuracy: 0.8515 (+/- 0.1790)
  Fold scores: [0.6777 0.9987 0.9936 0.9956 0.5918]
Regressor CV MAE: 2.8086 (+/- 1.4092)
  Fold scores: [0.9137 1.5958 2.8335 4.1086 4.591 ]

Training final models with best hyperparameters...
Regression MAE (Detour km): 2.74
Classification Accuracy: 0.94
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95      4327
           1       0.89      0.99      0.94      3350

    accuracy                           0.94      7677
   macro avg       0.94      0.95      0.94      7677
weighted avg       0.95      0.94      0.94      7677


Regression RMSE (Detour km): 5.4127
Classification F1: 0.9389
Classification ROC-AUC: 0.9819
Saved figures/feature_importance.png
Saved figures/roc_curve.png

Running feature ablation study (ROC-AUC drop)...


ROC-AUC drop when feature removed (higher = more important):
  overlap_ratio: +0.0102
  fare_ratio: +0.0028
  shareability_count: +0.0007
  driver_orig_dist: +0.0001
  driver_approx_fare: -0.0000
  rider_approx_fare: +0.0000
  pickup_hour: +0.0000
  pickup_dow: +0.0000
  time_diff_mins: -0.0002
Saved figures/ablation_study.png

Generating learning curves...


/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
2 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/sklearn.py", line 1761, in fit
    raise Valu

Saved figures/learning_curve.png

Running system-level bipartite matching evaluation...



System-Level Results (Bipartite Matching on Test Set):
  Unique test drivers  : 3969
  Optimal matches      : 3926
  Match rate           : 98.9%
  Avg driver detour    : 14.65 km
  Avg driver earnings  : +$5.75 per ride
  Avg rider savings    : -$5.75 per ride
  Avg CO2 saved        : 3098 g per match


Saved figures/system_tradeoff_curve.png
